# 04 Forecasting: simple baselines vs LightGBM

Forecasts weekly sales one week ahead for all 9,147 Texas series and compares three simple methods against one LightGBM model. Error is broken down by demand pattern, store, and category to see where the complex model earns its keep.

Inputs: `tx_weekly.parquet` (02_join), `series_profile.parquet` (03_eda). Output: `test_predictions.parquet`, used by the cost analysis and the dashboard.

**Design decisions**
- One week ahead, every week. Assumes weekly replenishment.
- Time-based split: train on weeks 1 to 251, validate on 252 to 264, test on the last 13 weeks (265 to 277). Never random, because the future can't train the past.
- Simple methods: last week's sales (naive), 4-week average, 13-week average. The best one is chosen on validation, not test.
- LightGBM: one global model across all series, Tweedie objective (suited to counts with many zeros).
- Price, stocking days, SNAP days, and event days for the forecast week are treated as known in advance. Walmart sets its own prices and the calendar is public.

## Kernel check

In [1]:
import sys, pandas, duckdb, pyarrow, lightgbm
print(sys.executable)
print(pandas.__version__, duckdb.__version__, pyarrow.__version__, lightgbm.__version__)

/Users/rajkandula/projects/walmart-demand-forecasting/.venv/bin/python
3.0.6 1.5.5 25.0.1 4.7.0


## Setup

In [2]:
import sys; sys.path.append("..")
import duckdb
import numpy as np
import pandas as pd
import lightgbm as lgb
from src.config import PROCESSED_DIR

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE VIEW weekly AS SELECT * FROM read_parquet('{PROCESSED_DIR / 'tx_weekly.parquet'}')")
con.sql(f"CREATE OR REPLACE VIEW profile AS SELECT * FROM read_parquet('{PROCESSED_DIR / 'series_profile.parquet'}')")

## Features with window functions

Each row is one series in one week. `LAG(sales, k)` is the series' sales k weeks earlier. The rolling averages use `ROWS BETWEEN n PRECEDING AND 1 PRECEDING`, so they never include the week being forecast. Only active weeks (on or after the first priced week) are kept, matching 03_eda.

In [3]:
con.sql("""
CREATE OR REPLACE TABLE features AS
WITH weeks AS (
    SELECT wm_yr_wk, ROW_NUMBER() OVER (ORDER BY wm_yr_wk) AS week_num
    FROM (SELECT DISTINCT wm_yr_wk FROM weekly)
),
base AS (
    SELECT
        w.id, w.item_id, w.dept_id, w.cat_id, w.store_id,
        w.wm_yr_wk, w.week_start, k.week_num,
        w.sales, w.avg_price, w.days_priced, w.snap_days, w.event_days,
        p.pattern
    FROM weekly AS w
    JOIN weeks AS k ON w.wm_yr_wk = k.wm_yr_wk
    JOIN profile AS p ON w.id = p.id
    WHERE w.wm_yr_wk >= p.first_priced_week
)
SELECT
    *,
    LAG(sales, 1)  OVER (PARTITION BY id ORDER BY week_num) AS lag_1,
    LAG(sales, 2)  OVER (PARTITION BY id ORDER BY week_num) AS lag_2,
    LAG(sales, 4)  OVER (PARTITION BY id ORDER BY week_num) AS lag_4,
    LAG(sales, 52) OVER (PARTITION BY id ORDER BY week_num) AS lag_52,
    AVG(sales) OVER (PARTITION BY id ORDER BY week_num
                     ROWS BETWEEN 4 PRECEDING AND 1 PRECEDING) AS ma_4,
    AVG(sales) OVER (PARTITION BY id ORDER BY week_num
                     ROWS BETWEEN 13 PRECEDING AND 1 PRECEDING) AS ma_13,
    AVG(avg_price) OVER (PARTITION BY id ORDER BY week_num
                         ROWS BETWEEN 13 PRECEDING AND 1 PRECEDING) AS price_ma_13,
    WEEKOFYEAR(week_start) AS week_of_year,
    MONTH(week_start) AS month
FROM base
""")

n_rows, n_series, max_week, n_units = con.sql(
    "SELECT COUNT(*), COUNT(DISTINCT id), MAX(week_num), SUM(sales) FROM features"
).fetchone()
print(n_rows, n_series, max_week, n_units)

assert n_rows == 2_046_387     # active weekly rows from 03_eda
assert n_series == 9_147
assert max_week == 277
assert n_units == 19_199_909

2046387 9147 277 19199909


## Load and split by time

Validation picks the best simple method and stops LightGBM training early. Test is touched only once, for the final comparison.

In [4]:
df = con.sql("SELECT * FROM features").df()
for c in ["store_id", "cat_id", "dept_id", "pattern"]:
    df[c] = df[c].astype("category")
df["price_ratio"] = df["avg_price"] / df["price_ma_13"]
df["naive"] = df["lag_1"]

VALID_START, TEST_START = 252, 265
train = df[df.week_num < VALID_START]
valid = df[(df.week_num >= VALID_START) & (df.week_num < TEST_START)].copy()
test = df[df.week_num >= TEST_START].copy()

print(len(train), len(valid), len(test))
print(test.week_num.min(), test.week_num.max(), test.week_num.nunique())
assert len(train) + len(valid) + len(test) == len(df)
assert test.week_num.nunique() == 13

1808580 118896 118911
265 277 13


## Best simple method, chosen on validation

A brand-new item with no history gets a forecast of 0 from the simple methods (`fillna(0)`).

In [5]:
SIMPLE = ["naive", "ma_4", "ma_13"]
for d in (valid, test):
    d[SIMPLE] = d[SIMPLE].fillna(0)

valid_mae = {m: round((valid[m] - valid["sales"]).abs().mean(), 4) for m in SIMPLE}
best_simple = min(valid_mae, key=valid_mae.get)
print(valid_mae)
print("best simple method:", best_simple)

{'naive': np.float64(3.5657), 'ma_4': np.float64(3.3033), 'ma_13': np.float64(3.6769)}
best simple method: ma_4


## Train LightGBM

Early stopping halts training once validation error stops improving for 100 rounds, which guards against overfitting. Takes a few minutes.

In [6]:
FEATURES = ["lag_1", "lag_2", "lag_4", "lag_52", "ma_4", "ma_13",
            "avg_price", "price_ratio", "days_priced", "snap_days", "event_days",
            "week_of_year", "month", "store_id", "cat_id", "dept_id", "pattern"]

model = lgb.LGBMRegressor(
    objective="tweedie", metric="l1",
    n_estimators=1000, learning_rate=0.05, num_leaves=63,
    min_child_samples=50, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, random_state=42, verbose=-1,
)
model.fit(
    train[FEATURES], train["sales"],
    eval_set=[(valid[FEATURES], valid["sales"])],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)
test["lgbm"] = model.predict(test[FEATURES]).clip(0)
print("best iteration:", model.best_iteration_)

/Users/rajkandula/projects/walmart-demand-forecasting/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's l1: 3.0735
[200]	valid_0's l1: 3.03673
[300]	valid_0's l1: 3.02298
[400]	valid_0's l1: 3.01838
[500]	valid_0's l1: 3.01707
[600]	valid_0's l1: 3.01399
[700]	valid_0's l1: 3.01167
[800]	valid_0's l1: 3.01154
Early stopping, best iteration is:
[751]	valid_0's l1: 3.01091
best iteration: 751


## Feature importance

Which inputs the model leaned on most (gain = total error reduction from splits on that feature). An interviewer will ask what drives the forecast.

In [7]:
importance = (
    pd.Series(model.booster_.feature_importance("gain"), index=FEATURES)
    .sort_values(ascending=False)
)
print((importance / importance.sum()).round(3).head(10))

ma_4            0.633
lag_1           0.224
ma_13           0.091
avg_price       0.010
lag_52          0.008
pattern         0.007
week_of_year    0.006
price_ratio     0.005
lag_2           0.005
dept_id         0.004
dtype: float64


## Save test and validation predictions in long format

One row per series, week, and method. The cost analysis reads both files: validation errors set each series' safety stock, and test weeks are where costs are measured. The dashboard reads the test file.

Caveat: LightGBM used the validation weeks for early stopping, so its validation errors are slightly optimistic. That means slightly less safety stock for LightGBM in the cost analysis, which works against it, not for it.

In [8]:
keep = ["id", "item_id", "dept_id", "cat_id", "store_id", "pattern",
        "wm_yr_wk", "week_num", "sales", "avg_price", "price_ma_13"]
valid["lgbm"] = model.predict(valid[FEATURES]).clip(0)

def to_long(d):
    out = d[keep + SIMPLE + ["lgbm"]].melt(
        id_vars=keep, var_name="method", value_name="forecast"
    )
    out["is_best_simple"] = out["method"] == best_simple
    return out

preds = to_long(test)
valid_preds = to_long(valid)
preds.to_parquet(PROCESSED_DIR / "test_predictions.parquet", index=False)
valid_preds.to_parquet(PROCESSED_DIR / "valid_predictions.parquet", index=False)

print(preds.shape, valid_preds.shape)
assert len(preds) == 4 * len(test)
assert len(valid_preds) == 4 * len(valid)

(475644, 14) (475584, 14)


## Score by segment

- **WMAPE**: total absolute error divided by total units. Comparable across segments of very different size.
- **MAE / RMSE**: average error per series-week; RMSE punishes big misses more.
- **Bias**: total forecast minus total actual, over total actual. Positive means over-forecasting (overstock risk), negative means under-forecasting (stockout risk).

In [9]:
score_sql = """
SELECT {by} method,
       ROUND(SUM(ABS(forecast - sales)) / SUM(sales), 3) AS wmape,
       ROUND(AVG(ABS(forecast - sales)), 3) AS mae,
       ROUND(SQRT(AVG(POWER(forecast - sales, 2))), 3) AS rmse,
       ROUND(SUM(forecast - sales) / SUM(sales), 3) AS bias
FROM preds
GROUP BY ALL
ORDER BY {by} wmape
"""
con.sql(score_sql.format(by="")).show()
con.sql(score_sql.format(by="pattern,")).show()
con.sql(score_sql.format(by="store_id,")).show()
con.sql(score_sql.format(by="cat_id,")).show()

┌─────────┬────────┬────────┬────────┬────────┐
│ method  │ wmape  │  mae   │  rmse  │  bias  │
│ varchar │ double │ double │ double │ double │
├─────────┼────────┼────────┼────────┼────────┤
│ lgbm    │  0.342 │  2.955 │  6.237 │ -0.019 │
│ ma_4    │  0.363 │  3.144 │  6.818 │ -0.007 │
│ naive   │  0.402 │  3.475 │  7.037 │ -0.003 │
│ ma_13   │  0.402 │  3.483 │  7.821 │ -0.036 │
└─────────┴────────┴────────┴────────┴────────┘

┌────────────────────────────────────────────────────┬─────────┬────────┬────────┬────────┬────────┐
│                      pattern                       │ method  │ wmape  │  mae   │  rmse  │  bias  │
│ enum('erratic', 'intermittent', 'lumpy', 'smooth') │ varchar │ double │ double │ double │ double │
├────────────────────────────────────────────────────┼─────────┼────────┼────────┼────────┼────────┤
│ erratic                                            │ lgbm    │  0.405 │  4.249 │  8.653 │ -0.023 │
│ erratic                                            │ ma_4   

## Results

**Split.** 1,808,580 training rows (weeks 1 to 251), 118,896 validation rows (252 to 264), 118,911 test rows (265 to 277).

**Best simple method.** The 4-week moving average, chosen on validation (MAE 3.30 units per series-week, vs 3.57 for naive and 3.68 for the 13-week average). Recent weeks predict better than a long average, but a single week is too noisy.

**LightGBM.** Early stopping at 596 rounds (validation MAE 3.01).

**Test results, all 9,147 series:**

| Method | WMAPE | MAE | RMSE | Bias |
|---|---|---|---|---|
| LightGBM | 0.341 | 2.955 | 6.226 | -1.8% |
| 4-week average | 0.363 | 3.144 | 6.818 | -0.7% |
| 13-week average | 0.402 | 3.483 | 7.821 | -3.6% |
| Naive | 0.402 | 3.475 | 7.037 | -0.3% |

LightGBM cuts WMAPE by 6.1% relative to the 4-week average (0.363 to 0.341) and RMSE by 8.7%, so it helps most on large misses.

**LightGBM vs 4-week average, relative WMAPE improvement:**

| Segment | 4-wk avg | LightGBM | Improvement |
|---|---|---|---|
| Smooth | 0.314 | 0.297 | 5.4% |
| Erratic | 0.429 | 0.405 | 5.6% |
| Intermittent | 0.488 | 0.450 | 7.8% |
| Lumpy | 0.590 | 0.557 | 5.6% |
| FOODS | 0.336 | 0.311 | 7.4% |
| HOUSEHOLD | 0.386 | 0.369 | 4.4% |
| HOBBIES | 0.490 | 0.472 | 3.7% |
| TX_1 | 0.377 | 0.358 | 5.0% |
| TX_2 | 0.352 | 0.328 | 6.8% |
| TX_3 | 0.364 | 0.341 | 6.3% |

LightGBM wins every segment, but the margin is modest everywhere and smallest in HOBBIES. Percentage gains are similar across patterns, so accuracy alone doesn't say where the model is worth running. Smooth series carry 73% of volume, so the same percentage gain there moves far more units. The cost analysis turns this into dollars.

**Bias.** LightGBM under-forecasts total demand by 1.8%, more than the 4-week average (0.7%). Under-forecasting means stockouts, so this matters in the cost stage.

**What drives the model.** By gain: 4-week average 64%, last week's sales 22%, 13-week average 9%. Everything else (other lags, price, SNAP, events, calendar, store, category) adds 4.9% combined. At weekly level the model is mostly a smarter blend of recent history; price and event effects are small.